# Phase 0 — Retroactive validation

**Goal:** test whether MP + inferred Order Flow + Context features can retrospectively
identify the trades in the FY25-FY26 Zerodha log that would have been better off skipped.

**Verdict criteria** (pre-committed in `evaluation/phase0.py`, do not change):

1. Skip-accuracy on bottom-decile losers ≥ 65%
2. Net P&L improvement (counterfactual) ≥ 30%
3. Retained-trades Sharpe ≥ 1.5× full-series Sharpe
4. `pytest tests/test_no_leakage.py` passes

All four must hold. This notebook runs the pipeline end-to-end and prints the verdict.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from pathlib import Path

from nomad_sniper.utils.settings import settings
from nomad_sniper.data.trades import load_zerodha_trades
from nomad_sniper.data.round_trips import pair_round_trips
from nomad_sniper.data.bars import load_minute_bars, UNDERLYINGS
from nomad_sniper.features.pipeline import build_features_for_trades
from nomad_sniper.labels.actual_trades import label_actual_trades
from nomad_sniper.labels.cost_model import ZerodhaFnoCostModel
from nomad_sniper.models.lightgbm_skip import train_skip_classifier
from nomad_sniper.evaluation.splits import walk_forward
from nomad_sniper.evaluation.metrics import (
    skip_accuracy_by_quality_bucket,
    counterfactual_pnl,
    sharpe_ratio,
    daily_pnl_series,
)
from nomad_sniper.evaluation.phase0 import run_phase0_verdict

pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. Load and pair trades

Expected input: `data/raw/zerodha_trades_fy25_fy26.csv` (export from console.zerodha.com → Reports → Trades).

In [ ]:
csv_path = settings.raw_dir / 'zerodha_trades_fy25_fy26.csv'
trades = load_zerodha_trades(csv_path)
round_trips = pair_round_trips(trades)
print(f'{len(trades):,} legs → {len(round_trips):,} round trips')
print(f'Date range: {min(r.entry_at for r in round_trips).date()} → {max(r.exit_at for r in round_trips).date()}')

## 2. Label round trips with net P&L

Slippage starts at the ₹0.10/share placeholder. **Calibrate this against your own fills**
(midprice-at-decision vs actual fill, averaged) before treating verdict numbers as final.

In [ ]:
cost_model = ZerodhaFnoCostModel(slippage_inr_per_share=0.10)
labels = label_actual_trades(round_trips, cost_model=cost_model)
labels.head()

In [ ]:
summary = pd.Series({
    'Trades':            len(labels),
    'Gross P&L (₹)':     labels['gross_pnl'].sum(),
    'Total cost (₹)':    labels['total_cost'].sum(),
    'Net P&L (₹)':       labels['net_pnl'].sum(),
    'Win rate':          labels['is_winner'].mean(),
    'Profit factor':     labels.loc[labels['net_pnl']>0,'net_pnl'].sum()
                          / abs(labels.loc[labels['net_pnl']<0,'net_pnl'].sum()),
})
summary

## 3. Build features

Needs `upstox_<underlying>_fut_<YYYYMMDD>.parquet` files in `data/raw/`.

In [ ]:
bars_by_underlying = {}
for u in UNDERLYINGS:
    try:
        bars_by_underlying[u] = load_minute_bars(u)
        print(f'{u}: {len(bars_by_underlying[u]):,} bars')
    except FileNotFoundError as e:
        print(f'  skipping {u}: {e}')

In [ ]:
def infer_underlying(symbol):
    s = symbol.upper()
    if s.startswith('BANKNIFTY'): return 'banknifty'
    if s.startswith('FINNIFTY'):  return 'finnifty'
    if s.startswith('NIFTY'):     return 'nifty'
    return None

entries = []
for rt in round_trips:
    u = infer_underlying(rt.symbol)
    if u and u in bars_by_underlying:
        entries.append((rt.entry_trade_id, rt.entry_at, u))

features = build_features_for_trades(entries, bars_by_underlying)
print(f'Feature matrix shape: {features.shape}')
features.head()

## 4. Walk-forward training

6-month train → 1-month test, advancing 1 month per fold, with a 2-day purge gap.

In [ ]:
common = features.index.intersection(labels.index)
X = features.loc[common]
y = labels.loc[common, 'is_winner']
decision_times = pd.to_datetime(X['decision_time'])

cat_cols = [c for c in ('location_vs_prev_value', 'open_location', 'time_of_day_bucket', 'underlying') if c in X.columns]

oos_rows = []
for split in walk_forward(decision_times, train_months=6, test_months=1, purge_days=2):
    train_mask = split.train_mask(decision_times)
    test_mask  = split.test_mask(decision_times)
    if train_mask.sum() < 50 or test_mask.sum() < 5:
        continue
    clf = train_skip_classifier(
        X.loc[train_mask], y.loc[train_mask],
        categorical_features=cat_cols,
    )
    proba = clf.predict_proba_take(X.loc[test_mask])
    for tid, p in zip(X.loc[test_mask].index, proba):
        oos_rows.append({'trade_id': tid, 'p_take': float(p),
                         'test_end': split.test_end})

oos = pd.DataFrame(oos_rows).set_index('trade_id')
print(f'OOS predictions: {len(oos)} trades across walk-forward folds')

## 5. Verdict

In [ ]:
import subprocess
leakage_result = subprocess.run(
    ['pytest', '-q', 'tests/test_no_leakage.py'],
    capture_output=True, text=True,
)
leakage_passed = leakage_result.returncode == 0
print(f'Leakage tests: {"PASS" if leakage_passed else "FAIL"}')
if not leakage_passed:
    print(leakage_result.stdout)

In [ ]:
skip = (oos['p_take'] < 0.5).astype(int)

verdict = run_phase0_verdict(
    labels=labels,
    skip_decisions=skip,
    leakage_tests_passed=leakage_passed,
)

print(f'\n=== Phase 0 verdict: {verdict.verdict.upper()} ===')
print(f'  Skip-accuracy bottom decile:  {verdict.skip_accuracy_bottom_decile:.1%}  (need ≥ 65%)')
print(f'  Net P&L improvement:          {verdict.net_pnl_improvement_pct:+.1f}%  (need ≥ 30%)')
print(f'  Sharpe uplift:                {verdict.sharpe_uplift:.2f}x  (need ≥ 1.5x)')
print(f'  Leakage tests passed:         {verdict.leakage_tests_passed}')
if verdict.reasons:
    print('\nWhy not GO:')
    for r in verdict.reasons:
        print(f'  • {r}')

## 6. Diagnostic: skip behaviour by P&L decile

If healthy: skip rate is high in decile 1 (worst losers) and declines monotonically toward decile 10.

In [ ]:
bucket = skip_accuracy_by_quality_bucket(labels, skip, bucket_col='pnl_decile')
bucket

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
bucket['skip_rate'].plot(kind='bar', ax=ax)
ax.set_xlabel('P&L decile (1 = worst losers, 10 = best winners)')
ax.set_ylabel('Model skip rate')
ax.set_title('Skip rate by P&L decile — should slope down left-to-right')
ax.axhline(0.65, color='red', linestyle='--', label='65% threshold for decile 1')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Cost sensitivity

How robust is the verdict to slippage assumptions? Sweep across 0.5x – 3x baseline.

In [ ]:
rows = []
for mult in (0.5, 1.0, 1.5, 2.0, 3.0):
    cm = ZerodhaFnoCostModel(slippage_inr_per_share=0.10 * mult)
    lbl = label_actual_trades(round_trips, cost_model=cm)
    cf = counterfactual_pnl(lbl, skip)
    rows.append({'slippage_x': mult, **cf})
pd.DataFrame(rows)[['slippage_x','actual_total_pnl','counterfactual_total_pnl','improvement_pct']]